In [ ]:
#aggregation

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE6_T1","PLATE7_T1"]

# Paths for the three separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"27march", "antibiotics_aggregated_wells_median.csv")
OUTPUT_CSV_MEAN   = os.path.join(PROJECT_ROOT,"27march", "antibiotics_aggregated_wells_mean.csv")
OUTPUT_CSV_STD    = os.path.join(PROJECT_ROOT,"27march", "antibiotics_aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_mean = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate aggregations
            well_median = np.median(all_cells_in_well, axis=0)
            well_mean   = np.mean(all_cells_in_well, axis=0)
            well_std    = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Efficiently map feature indices to values
            feat_cols = {idx: val for idx, val in enumerate(well_median)}
            all_plates_median.append({**base_info, **feat_cols})
            
            feat_cols_mean = {idx: val for idx, val in enumerate(well_mean)}
            all_plates_mean.append({**base_info, **feat_cols_mean})
            
            feat_cols_std = {idx: val for idx, val in enumerate(well_std)}
            all_plates_std.append({**base_info, **feat_cols_std})

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_mean   = pd.DataFrame(all_plates_mean)
df_std    = pd.DataFrame(all_plates_std)

# Helper function to reorder columns consistently
def reorder_cols(df):
    if df.empty: return df
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = sorted([c for c in df.columns if c not in meta_cols])
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_mean   = reorder_cols(df_mean)
df_std    = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_mean.to_csv(OUTPUT_CSV_MEAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved: {OUTPUT_CSV_MEDIAN}")
print(f"Mean data saved:   {OUTPUT_CSV_MEAN}")
print(f"Std Dev data saved: {OUTPUT_CSV_STD}")

In [ ]:
#patch count

In [ ]:
import numpy as np
import os
from tqdm import tqdm

PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE6_T1","PLATE7_T1"]

total_patches = 0
total_files = 0

print("Calculating total patch count...")

for plate in PLATES:
    feature_path = os.path.join(PROJECT_ROOT, "features", plate)
    
    if not os.path.exists(feature_path):
        print(f"Skipping {plate}: Path not found.")
        continue
        
    # Walk through all well subfolders
    for root, dirs, files in os.walk(feature_path):
        for file in files:
            if file.endswith(".npz"):
                file_path = os.path.join(root, file)
                try:
                    with np.load(file_path) as data:
                        # 'features' is the standard key in DeepProfiler npz files
                        # We only need the shape[0] (number of rows/cells)
                        total_patches += data["features"].shape[0]
                        total_files += 1
                except Exception as e:
                    print(f"Could not read {file}: {e}")

print("\n--- Final Statistics ---")
print(f"Total .npz files (sites) processed: {total_files}")
print(f"Total number of patches (cells):     {total_patches:,}")

In [ ]:
#remove less than 5 patches

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_DIR = os.path.join(PROJECT_ROOT, "27march")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_filtered")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# List of the aggregation files you created in the previous step
FILES_TO_FILTER = [
    "antibiotics_aggregated_wells_median.csv",
    "antibiotics_aggregated_wells_mean.csv",
    "antibiotics_aggregated_wells_std.csv"
]

# 3. PROCESSING LOOP
for file_name in FILES_TO_FILTER:
    file_path = os.path.join(INPUT_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {file_name}: File not found.")
        continue

    print(f"\n--- Processing {file_name} ---")
    df = pd.read_csv(file_path)

    # Identify the wells to keep and remove
    # We use 'Cell_Count' which we added during the aggregation step
    filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()
    removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

    # 4. REPORT
    print(f"Original wells: {len(df)}")
    print(f"Wells kept:     {len(filtered_df)}")
    print(f"Wells removed:  {len(removed_df)}")

    if not removed_df.empty and "median" in file_name:
        # Just show the list once (for the median file) to avoid clutter
        print(f"\nExample of removed wells (Count < {STRICT_THRESHOLD}):")
        print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].head(10).to_string(index=False))

    # 5. SAVE DATA
    # Renaming the output to include the 'min5' suffix
    output_name = file_name.replace(".csv", "_min5.csv")
    output_path = os.path.join(OUTPUT_DIR, output_name)
    filtered_df.to_csv(output_path, index=False)
    
    print(f"Filtered data saved to: {output_path}")

print("\nAll filtering tasks complete!")

In [ ]:
# all median

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "27march_filtered", "antibiotics_aggregated_wells_median_min5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_2", "UMAP_all_median")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# 2. PRE-PROCESSING
# Extract Base Treatment (e.g., 'vancomycin')
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

# Define Symbol and special Color logic
def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        # Controls get a unique label so we can color them black
        color_group = "Control (nosgrna)"
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    else:
        # Mutants use their antibiotic name for color
        color_group = row['Treatment_Base']
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
        
    return pd.Series([color_group, symbol])

df[['Color_Group', 'Symbol_Type']] = df.apply(assign_plot_logic, axis=1)

# Create a color map to force Controls to be Black
unique_treatments = df['Color_Group'].unique()
color_map = {t: px.colors.qualitative.Alphabet[i % 26] for i, t in enumerate(unique_treatments)}
color_map["Control (nosgrna)"] = "#000000"  # Hex for pure black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='Color_Group',
        symbol='Symbol_Type',
        color_discrete_map=color_map, # Forces control to black
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Cell_Count': True},
        title=f"UMAP: {config['name']} (Black Controls: Dot=P6, X=P7)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls slightly larger and fully opaque to stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Treatments & Controls')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_BlackControls_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#all mean

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "27march_filtered", "antibiotics_aggregated_wells_mean_min5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_2", "UMAP_all_mean")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# 2. PRE-PROCESSING
# Extract Base Treatment (e.g., 'vancomycin')
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

# Define Symbol and special Color logic
def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        # Controls get a unique label so we can color them black
        color_group = "Control (nosgrna)"
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    else:
        # Mutants use their antibiotic name for color
        color_group = row['Treatment_Base']
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
        
    return pd.Series([color_group, symbol])

df[['Color_Group', 'Symbol_Type']] = df.apply(assign_plot_logic, axis=1)

# Create a color map to force Controls to be Black
unique_treatments = df['Color_Group'].unique()
color_map = {t: px.colors.qualitative.Alphabet[i % 26] for i, t in enumerate(unique_treatments)}
color_map["Control (nosgrna)"] = "#000000"  # Hex for pure black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='Color_Group',
        symbol='Symbol_Type',
        color_discrete_map=color_map, # Forces control to black
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Cell_Count': True},
        title=f"UMAP: {config['name']} (Black Controls: Dot=P6, X=P7)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls slightly larger and fully opaque to stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Treatments & Controls')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_BlackControls_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#feature seleciton

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "27march_filtered", "antibiotics_aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_filtered")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Selection based on batch noise ranking
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
# Removes features that are flat/near-zero across the whole experiment first
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 3000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=3000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_27march.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Active features (Step 0): {len(active_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#combineren van antibitiocs with the knockdown

In [ ]:
import pandas as pd

# Define file paths
file1_path = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal\27march_filtered\antibiotics_aggregated_wells_median_min5.csv'
file2_path = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal\10marchecht\vettedcellcounts_10march.csv'
output_path = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal\27march\combined_data.csv'

# Load the files
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# Identify common columns
# This ensures we only paste data where the feature names match exactly
common_cols = df2.columns.intersection(df1.columns)

print(f"Columns being matched: {list(common_cols)}")

# Filter File 1 to only include columns present in File 2
df1_filtered = df1[common_cols]

# Concatenate File 1 below File 2
combined_df = pd.concat([df2, df1_filtered], axis=0, ignore_index=True)

# Save the result
combined_df.to_csv(output_path, index=False)
print(f"Successfully saved combined file to: {output_path}")

In [ ]:
#combining knockdown and antibiotics

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP & PATHS ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file1_path = os.path.join(PROJECT_ROOT, "27march_filtered", "antibiotics_aggregated_wells_median_min5.csv")
file2_path = os.path.join(PROJECT_ROOT, "10marchecht", "vettedcellcounts_10march.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_2", "Combined_UMAP_Channels")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD AND MERGE DATA ---
print("Loading data...")
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)
anno_df = pd.read_excel(anno_path)

df1.columns = [str(c) for c in df1.columns]
df2.columns = [str(c) for c in df2.columns]

# Find common numerical features
feat_cols_1 = [c for c in df1.columns if c.isdigit()]
feat_cols_2 = [c for c in df2.columns if c.isdigit()]
common_features = sorted(list(set(feat_cols_1) & set(feat_cols_2)), key=int)

metadata = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
df1_sub = df1[metadata + common_features].copy()
df2_sub = df2[metadata + common_features].copy()
df1_sub['Source'] = 'Antibiotics_File'
df2_sub['Source'] = 'Pathway_File'

df_combined = pd.concat([df2_sub, df1_sub], axis=0, ignore_index=True)
df_combined = df_combined.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# --- 3. ANNOTATION & COLOR LOGIC ---
def create_final_annotation(row):
    treat = str(row['Treatment']).lower()
    if any(ctrl in treat for ctrl in ["no_sgrna", "nosgrna"]):
        return "Control (no_sgRNA)"
    if row['Source'] == 'Antibiotics_File':
        return str(row['Treatment']).split('_')[0]
    pathway = row['SubtiWiki Annotation 4'] if pd.notna(row['SubtiWiki Annotation 4']) else row['SubtiWiki Annotation 3']
    return pathway if pd.notna(pathway) else "Unknown/Other"

df_combined['Final_Label'] = df_combined.apply(create_final_annotation, axis=1)

# Generate Golden Angle Palette
all_cats = sorted([c for c in df_combined['Final_Label'].unique() if c not in ["Control (no_sgRNA)", "Unknown/Other"]])
golden_ratio_conjugate = 0.618033988749895
base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
h_values = [(i * golden_ratio_conjugate) % 1 for i in range(len(all_cats))]
color_map = {cat: base_palette[int(h * 255)] for i, (cat, h) in enumerate(zip(all_cats, h_values))}
color_map["Control (no_sgRNA)"] = "#000000"
color_map["Unknown/Other"] = "#222222"

# --- 4. CHANNEL DEFINITIONS ---
def get_channel_features(start, end):
    return [f for f in common_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Common_Features", "indices": common_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# --- 5. EXECUTION LOOP ---
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']}: No overlapping features found.")
        continue
    
    print(f"Processing UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and UMAP
    X_scaled = StandardScaler().fit_transform(df_combined[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_combined['UMAP1'], df_combined['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_combined['Symbol'] = df_combined['Source'].map({'Pathway_File': 'circle', 'Antibiotics_File': 'diamond'})

    # Create Plot
    fig = px.scatter(
        df_combined, x='UMAP1', y='UMAP2',
        color='Final_Label',
        symbol='Symbol',
        symbol_map={"circle": "circle", "diamond": "diamond"},
        hover_name='Treatment',
        hover_data=['Plate', 'Well_ID', 'Source'],
        title=f"Combined UMAP - {config['name'].replace('_', ' ')}<br><sup>Diamonds: Antibiotics (Big) | Circles: Pathways (Small)</sup>",
        color_discrete_map=color_map,
        template='plotly_white'
    )

    # Styling
    fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0.5, color='white')))
    
    # Specific Sizing: Diamonds BIG, Circles Standard
    fig.update_traces(marker=dict(size=12), selector=dict(marker_symbol='diamond'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='circle'))
    
    # Ensure Controls are black and opaque
    fig.for_each_trace(lambda t: t.update(marker=dict(color='black', opacity=1.0)) if "Control" in t.name else ())

    fig.update_layout(width=1300, height=850)
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f"Combined_UMAP_{config['name']}.html")
    fig.write_html(save_path)
    print(f"Saved to: {save_path}")

print("All channel plots complete.")

In [ ]:
#combined feature selection knockdown and antibiotics

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# File 2 (Original Pathway Data)
file2_raw_path = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv") 
# File 1 (Antibiotic Data)
file1_raw_path = os.path.join(PROJECT_ROOT, "27march_filtered", "antibiotics_aggregated_wells_median_min5.csv")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_2", "Combined_Vetted_Data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
CONTROL_LABEL = "no_sgRNA" 

print("Loading and Merging raw data...")
df1 = pd.read_csv(file1_raw_path)
df2 = pd.read_csv(file2_raw_path)

# Harmonize column names and find intersection
df1.columns = [str(c) for c in df1.columns]
df2.columns = [str(c) for c in df2.columns]
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
common_features = sorted(list(set(df1.columns) & set(df2.columns) - set(metadata_cols)), key=lambda x: int(x) if x.isdigit() else 0)

# Merge rows
df_raw = pd.concat([df2[metadata_cols + common_features], df1[metadata_cols + common_features]], axis=0, ignore_index=True)
feature_cols = common_features

# ==========================================
# TOOLBOX: FUNCTIONS (Modified for T0 mapping)
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=3000):
    ctrls = df[df['Treatment'].str.contains(CONTROL_LABEL, case=False, na=False)]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    return within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()

def filter_across_plate_stability_combined(df, features, top_n_to_keep=200):
    df_temp = df.copy()
    # Logic: Treat T1 Antibiotic plates as T0 for control comparison
    # This allows the function to calculate 'noise' between File 1 and File 2 controls
    df_temp['Pseudo_Timepoint'] = df_temp['Plate'].apply(lambda x: 'T0' if '_T1' in str(x) else str(x)[-2:])
    
    ctrls = df_temp[df_temp['Treatment'].str.contains(CONTROL_LABEL, case=False, na=False)]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    # We check T0 (which now includes File 1's T1), T1 (original), and T2
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp) or (tp == 'T0' and '_T1' in p)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises: return features
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    return total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    return [f for f in features if f not in to_drop]

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE ---
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()

# --- STEP 1: WITHIN-PLATE ---
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=3000)

# --- STEP 2: ACROSS-PLATE (With File 1 mapped to T0) ---
step2_features = filter_across_plate_stability_combined(df_raw, step1_features, top_n_to_keep=200)

# --- STEP 3: REDUNDANCY ---
final_feature_list = filter_redundancy(df_raw[metadata_cols + step2_features], step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_raw[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vetted_combined_antibiotics_pathways.csv")
df_final.to_csv(output_path, index=False)

print(f"\nWORKFLOW COMPLETE")
print(f"Features matched between files: {len(common_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP & PATHS ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Use the VETTED file we just created in the previous step
vetted_data_path = os.path.join(PROJECT_ROOT, "27march_2", "Combined_Vetted_Data", "vetted_combined_antibiotics_pathways.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "27march_2", "Combined_UMAP_Vetted_Channels")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD DATA ---
print("Loading vetted data and annotations...")
df_combined = pd.read_csv(vetted_data_path)
anno_df = pd.read_excel(anno_path)

# Ensure columns are strings
df_combined.columns = [str(c) for c in df_combined.columns]

# Re-identify Source if not in CSV (CSV contains metadata + vetted features)
# If 'Source' wasn't saved in the CSV, we re-derive it from Plate names
if 'Source' not in df_combined.columns:
    # Antibiotics are usually Plate 6/7 T1 based on your previous logic
    df_combined['Source'] = df_combined['Plate'].apply(
        lambda x: 'Antibiotics_File' if 'PLATE6' in str(x) or 'PLATE7' in str(x) else 'Pathway_File'
    )

# Merge pathway annotations
df_combined = df_combined.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# --- 3. ANNOTATION & COLOR LOGIC ---
def create_final_annotation(row):
    treat = str(row['Treatment']).lower()
    if any(ctrl in treat for ctrl in ["no_sgrna", "nosgrna"]):
        return "Control (no_sgRNA)"
    if row['Source'] == 'Antibiotics_File':
        return str(row['Treatment']).split('_')[0]
    pathway = row['SubtiWiki Annotation 4'] if pd.notna(row['SubtiWiki Annotation 4']) else row['SubtiWiki Annotation 3']
    return pathway if pd.notna(pathway) else "Unknown/Other"

df_combined['Final_Label'] = df_combined.apply(create_final_annotation, axis=1)

# Generate Golden Angle Palette for maximum distinction
all_cats = sorted([c for c in df_combined['Final_Label'].unique() if c not in ["Control (no_sgRNA)", "Unknown/Other"]])
golden_ratio_conjugate = 0.618033988749895
base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
h_values = [(i * golden_ratio_conjugate) % 1 for i in range(len(all_cats))]
color_map = {cat: base_palette[int(h * 255)] for i, (cat, h) in enumerate(zip(all_cats, h_values))}
color_map["Control (no_sgRNA)"] = "#000000"
color_map["Unknown/Other"] = "#222222"

# --- 4. CHANNEL DEFINITIONS (Vetted features only) ---
vetted_features = [c for c in df_combined.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Features", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# --- 5. EXECUTION LOOP ---
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']}: No vetted features in this range.")
        continue
    
    print(f"Processing UMAP for: {config['name']} ({len(config['indices'])} vetted features)...")
    
    X_scaled = StandardScaler().fit_transform(df_combined[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_combined['UMAP1'], df_combined['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_combined['Symbol'] = df_combined['Source'].map({'Pathway_File': 'circle', 'Antibiotics_File': 'diamond'})

    fig = px.scatter(
        df_combined, x='UMAP1', y='UMAP2',
        color='Final_Label',
        symbol='Symbol',
        symbol_map={"circle": "circle", "diamond": "diamond"},
        hover_name='Treatment',
        hover_data=['Plate', 'Well_ID', 'Source', 'Cell_Count'],
        title=f"Vetted Combined UMAP - {config['name'].replace('_', ' ')}<br><sup>Diamonds: Antibiotics (Big) | Circles: Pathways (Small)</sup>",
        color_discrete_map=color_map,
        template='plotly_white'
    )

    # Specific Sizing and Styling
    fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0.5, color='white')))
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='diamond')) # Made even bigger
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='circle'))
    
    # Black Controls
    fig.for_each_trace(lambda t: t.update(marker=dict(color='black', opacity=1.0)) if "Control" in t.name else ())

    fig.update_layout(width=1400, height=850)
    
    save_path = os.path.join(OUTPUT_DIR, f"Vetted_UMAP_{config['name']}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")

print("\nAll Vetted channel plots are complete!")